Analyzing Peng et al. 2019 PDAC dataset — 57,000 cells from 24 tumors and 11 normal pancreas samples. Goal: QC, clustering, and UMAP visualization to identify cell populations in the tumor microenvironment

In [ ]:
#import data
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import scrublet as scr

In [ ]:
#load data
adata = sc.read('Peng2019_raw.h5ad')
print(adata)

In [ ]:
#Explore the metadta
adata.obs['CONDITION']
#how many cells are in each category


In [ ]:
adata.obs['CONDITION'].value_counts()


In [ ]:
adata.obs['Patient'].value_counts()


In [ ]:
adata.obs['Type'].value_counts()


In [ ]:
adata.obs['Cell_type'].value_counts()

In [ ]:
#Cell types broken by normal and tumor
pd.crosstab(adata.obs['Cell_type'], adata.obs['Type'])

Ductal cell type 2: 0 in normal, 11,315 in tumor. Zero normal. These are the cancer cells — they only exist in tumor samples. This makes perfect sense.
Ductal cell type 1: 7,671 in normal, 2,646 in tumor. Mostly normal tissue. These are the non-malignant ductal cells.
Macrophage: 559 normal, 4,802 tumor. Almost 9x more macrophages in tumor. The tumor is actively recruiting macrophages — these are likely the immunosuppressive M2 TAMs.
Fibroblast: 940 normal, 5,802 tumor. 6x more in tumor. The tumor is activating fibroblasts to build that dense stroma.
Stellate: 623 normal, 5,284 tumor. Same pattern — stellate cells are being activated by the tumor to become CAFs.
T cell: 44 normal, 3,616 tumor. More T cells in tumor, but remember — the tumor samples have way more total cells (41,986 vs 15,544). As a proportion, T cells are still relatively scarce compared to the immunosuppressive cells. And the question for Week 2 is whether those T cells are functional or exhausted.
B cell: 31 normal, 2,416 tumor. Similar pattern.
Acinar: 1,423 normal, 512 tumor. Mostly normal — acinar cells are normal pancreas tissue that gets destroyed as the tumor grows.

Now we start QC. 

In [ ]:
print(adata.var.head(10))
print("\n")
# Check if any SYMBOL starts with MT-

mtgenes=adata.var.index[adata.var.index.str.contains('^MT', case=False)]
print(mtgenes.tolist())

Mitochondrial genes were removed during upstream preprocessing by the Besca pipeline. Mitochondrial-based QC filtering was performed prior to this analysis.

In [ ]:
#QC filtering
#check genes are in the SYMBOL column
adata.var.head()
#total counts per cell
adata.var['mt'] = adata.var['SYMBOL'].str.startswith('MT-')
adata.var['mt'] = adata.var['SYMBOL'].str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, inplace=True)
#qc_vars["mt"] =calculte the percentage of counts per gene
#calculated automatically
#n_genes-by_counts=number of genes per cell
#total counts
#pct_counts_mt - mitrochondrial percentage per cell
#mitrochondrial percentage



adata.obs.head()


In [ ]:
#visualize the QC
#Violin plots- distribution of total counts and genes across all the cells
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts'], jitter =0.4)	
#Scatter plots--total counts vs genes detected 
# Scatter plot - total counts vs genes detected
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts')


Violet plot:
The n_genes_by_counts — most cells detect between 500-5,000 genes, with the bulk around 1,000-3,000. No extreme outliers, which suggests some QC was already done upstream.
The total_counts — most cells have between 5,000-30,000 counts, but there's a long tail going up to 100,000. Those very high-count cells at the top could be doublets.
Scatter plot:


In [ ]:
print(adata.obs['n_genes_by_counts'].describe())
print("\n")
print(adata.obs['total_counts'].describe())

In [ ]:
import psutil
print(f"Available RAM: {psutil.virtual_memory().available / 1e9:.1f} GB")
print(f"Total RAM: {psutil.virtual_memory().total / 1e9:.1f} GB")

I examined the distribution, compared it against standard recommendations from Luecken & Theis, and chose a threshold appropriate for my tissue type and dataset.

In [ ]:
import scrublet as scr
import scipy.sparse as sp

# Run Scrublet per sample to save memory
doublet_scores = np.zeros(adata.n_obs)
predicted_doublets = np.zeros(adata.n_obs, dtype=bool)

for sample in adata.obs['Patient'].unique():
    print(f"Processing {sample}...")
    mask = adata.obs['Patient'] == sample
    adata_sample = adata[mask]
    
    X = adata_sample.X
    if not sp.issparse(X):
        X = sp.csr_matrix(X)
    
    scrub = scr.Scrublet(X, expected_doublet_rate=0.06)
    scores, preds = scrub.scrub_doublets(min_counts=2, min_cells=3, n_prin_comps=20)
    
    doublet_scores[mask] = scores
    predicted_doublets[mask] = preds

adata.obs['doublet_score'] = doublet_scores
adata.obs['predicted_doublet'] = predicted_doublets
print(f"\nDetected {predicted_doublets.sum()} doublets out of {len(predicted_doublets)} cells")

When two cells are captured in the same droplet, the resulting "cell" has a transcriptomic profile that is an average of both. In clustering, doublets often appear as a cluster sitting between two real clusters on UMAP, or they contaminate a real cluster with markers from another cell type. Scrublet simulates synthetic doublets by averaging random pairs of cells in your data, then scores each real cell based on how similar it is to the simulated doublets. Cells above a threshold (typically 0.25, but check the bimodal distribution scrublet generates) are flagged and removed.

worked. 655 doublets detected out of 57,530 cells (about 1.1%). That's lower than the expected 6%, which makes sense because the upstream Besca preprocessing likely already removed the most obvious doublets.

In [ ]:
#Filter- Remove the doublets
#Before Filtering
print(f"Cells before filtering: {adata.n_obs}")
#Filter
adata = adata[adata.obs['predicted_doublet'] == False].copy()
#After Filtering
print(f"Cells after filtering: {adata.n_obs}")

In [ ]:
#Normalization
#Save the raw data
adata.raw = adata.copy()
#normalize
sc.pp.normalize_total(adata, target_sum=1e4)
#log transform- log compresses the range
sc.pp.log1p(adata)

In [ ]:
pip install --user scikit-misc

In [ ]:
#HVG
#find genes of high variance--> focus--> identify cell identity
#sc.pp.highly_variable_genes()
#picks top 2000 most informative genes
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat_v3')
print(f"Number of highly variable genes: {adata.var['highly_variable'].sum()}")

Why: 2,000 dimensions is too many for clustering algorithms to work efficiently. PCA finds the 50 axes that capture the most variation. PC1 might separate immune from epithelial. PC2 might separate T cells from macrophages.

In [ ]:
#PCA
sc.pp.pca(adata, n_comps=50)
print(adata.obsm['X_pca'].shape)

In [ ]:
adata.obsm['X_pca_harmony'] = np.array(harmony_out.Z_corr, dtype=np.float32)
print(f"Harmony corrected shape: {adata.obsm['X_pca_harmony'].shape}")

In [ ]:
#build cell neighborhood graph
sc.pp.neighbors(adata, use_rep = "X_pca_harmony")
#UMap
sc.tl.umap(adata)
#calcualtes x, y coordiantes for every cell

In [ ]:
#Clustering
#Leiden finds groups of cells that are more connected to each other than to the rest. resolution=0.5 controls how many clusters — higher means more clusters.
sc.tl.leiden(adata, resolution = 0.5)

In [ ]:
#Plot Umap
sc.pl.umap(adata, color = ['leiden', 'Type', 'Patient', 'Cell_type'], ncols=1)

In [ ]:
import anndata
anndata.settings.allow_write_nullable_strings = True
adata.write('Peng2019_processed.h5ad')
print("Saved successfully.")